In [0]:

# Configuração do ambiente
catalog = "meu_catalog"
schema = "default"
volume = "inputs"

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
bronze_schema = f"{catalog}.bronze"

print(f"Volume: {volume_path}")
print(f"Schema Bronze: {bronze_schema}")

In [0]:
# Conferir os arquivos disponíveis no Volume
display(dbutils.fs.ls(volume_path))

In [0]:
# Criar o database/schema da camada Bronze
# REGRAS DE NEGÓCIO: A camada Bronze deve existir como schema independente e receber os dados brutos sem transformação de conteúdo ou estrutura.

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {bronze_schema}
""")

print(f"Schema disponível: {bronze_schema}")

display(
    spark.sql("SHOW SCHEMAS IN meu_catalog")
)

In [0]:
# REGRAS DE NEGÓCIO: O reset controlado das tabelas evita misturar execuções de validação anteriores com uma nova carga. Em produção, a estratégia deve ser substituída pelo controle de execução adequado ao Workflow.

TABELAS_BRONZE_RESET = [
    "tb_movies_info", "tb_movies_financials", "tb_movies_metrics",
    "tb_credits_and_tags", "tb_movies_reviews", "tb_cotacao_dolar",
]

for tabela in TABELAS_BRONZE_RESET:
    spark.sql(f"DROP TABLE IF EXISTS {bronze_schema}.{tabela}")
    print(f"Removida (se existia): {bronze_schema}.{tabela}")

In [0]:
# movies_info_TMDB_IMDB.csv -> tabela tb_movies_info
# REGRAS DE NEGÓCIO: CSVs da Bronze são lidos sem inferência de tipos para preservar o conteúdo original. A única coluna adicionada é ingestion_datetime, exigida pelo escopo para rastrear o momento da ingestão.

from pyspark.sql.functions import current_timestamp

df_movies_info = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{volume_path}/movies_info_TMDB_IMDB.csv")
)

df_movies_info_bronze = (
    df_movies_info
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_info_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_info")
)

print(f"Tabela gravada: {bronze_schema}.tb_movies_info")

# Validação rápida: tb_movies_info

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_info
        LIMIT 10
    """)
)


In [0]:
# REGRAS DE NEGÓCIO: Dados financeiros permanecem brutos na Bronze; limpeza, tipagem e regras de validade são responsabilidade da Silver.
# movies_financials_IMDB_TMDB.csv -> tabela tb_movies_financials

df_movies_financials = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{volume_path}/movies_financials_IMDB_TMDB.csv")
)

df_movies_financials_bronze = (
    df_movies_financials
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_financials_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")
)

print(f"Tabela gravada: {bronze_schema}.tb_movies_financials")

# Validação rápida: tb_movies_financials

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_financials
        LIMIT 10
    """)
)

In [0]:
# REGRAS DE NEGÓCIO: A leitura permissiva preserva registros afetados por Column Shift/corrupção estrutural, permitindo que a Silver faça a tipagem segura sem interromper o pipeline.
# movies_metrics_IMDB_TMDB.csv -> tabela tb_movies_metrics
from pyspark.sql.types import StructType, StructField, StringType

colunas_metrics = ["id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes"]
schema_metrics = StructType(
    [StructField(c, StringType(), True) for c in colunas_metrics]
    + [StructField("_corrupt_record", StringType(), True)]
)

df_movies_metrics = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema_metrics)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(f"{volume_path}/movies_metrics_IMDB_TMDB.csv")
)

df_movies_metrics_bronze = (
    df_movies_metrics
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_metrics_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")
)

df_validacao_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")
total = df_validacao_metrics.count()
corrompidos = df_validacao_metrics.filter(df_validacao_metrics._corrupt_record.isNotNull()).count()
print(f"Tabela gravada: {bronze_schema}.tb_movies_metrics ({total} linhas, {corrompidos} sinalizadas em _corrupt_record)")


# Validação rápida: tb_movies_metrics

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_metrics
        LIMIT 10
    """)
)

In [0]:
# credits_and_tags_IMDB_TMDB.csv -> tabela tb_credits_and_tags
# Mesmo problema de aspas nao fechadas do arquivo de metrics, aqui vazando texto
# de sinopse/tagline nas colunas de genres/production_companies. Mesmo tratamento.

colunas_credits = ["id", "genres", "production_companies", "production_countries",
                   "spoken_languages", "keywords", "directors", "writers", "cast"]
schema_credits = StructType(
    [StructField(c, StringType(), True) for c in colunas_credits]
    + [StructField("_corrupt_record", StringType(), True)]
)

df_credits_and_tags = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema_credits)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(f"{volume_path}/credits_and_tags_IMDB_TMDB.csv")
)

df_credits_and_tags_bronze = (
    df_credits_and_tags
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_credits_and_tags_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")
)

df_validacao_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")
total = df_validacao_credits.count()
corrompidos = df_validacao_credits.filter(df_validacao_credits._corrupt_record.isNotNull()).count()
print(f"Tabela gravada: {bronze_schema}.tb_credits_and_tags ({total} linhas, {corrompidos} sinalizadas em _corrupt_record)")

# Validação rápida: tb_credits_and_tags

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_credits_and_tags
        LIMIT 10
    """)
)

In [0]:
# REGRAS DE NEGÓCIO: Avaliações entram sem limpeza de conteúdo na Bronze; duplicidade, escala de nota e comentários vazios são tratados somente na Silver.
# movies_reviews.csv -> tabela tb_movies_reviews

df_movies_reviews = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{volume_path}/movies_reviews.csv")
)

df_movies_reviews_bronze = (
    df_movies_reviews
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_reviews_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")
)

print(f"Tabela gravada: {bronze_schema}.tb_movies_reviews")

# Validação rápida: tb_movies_reviews

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_reviews
        LIMIT 10
    """)
)


In [0]:
from datetime import datetime, timedelta

# Valores padrão dos widgets
_data_fim_default = datetime.now()
_data_inicio_default = _data_fim_default - timedelta(days=7)

data_inicio_default = _data_inicio_default.strftime("%m-%d-%Y")
data_fim_default = _data_fim_default.strftime("%m-%d-%Y")

dbutils.widgets.text(
    "data_inicio",
    data_inicio_default,
    "Data início (MM-DD-AAAA)"
)

dbutils.widgets.text(
    "data_fim",
    data_fim_default,
    "Data fim (MM-DD-AAAA)"
)

# Ler os parâmetros dos widgets

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Data início: {data_inicio}")
print(f"Data fim: {data_fim}")

import json
import requests

# volume_path vem da célula de configuração do notebook
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)
print(url)

try:
    response = requests.get(url, timeout=30)
    print(f"Status HTTP: {response.status_code}")
    response.raise_for_status()   
    # erro HTTP (4xx/5xx) NÃO cai no fallback: falha visível
    dados_cotacao = response.json()
    print("Cotação obtida via API do BCB.")
except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
    # Só rede bloqueada pelo ambiente usa o fallback (JSON gerado por extração real da API).
    caminho = f"{volume_path}/cotacao_dolar_manual.json"
    print(f"AVISO: API inacessível ({e}). Usando fallback: {caminho}")
    with open(caminho, "r", encoding="utf-8") as f:
        dados_cotacao = json.load(f)
        from datetime import datetime
        ini = datetime.strptime(data_inicio, "%m-%d-%Y").date()
        fim = datetime.strptime(data_fim, "%m-%d-%Y").date()
        dados_cotacao["value"] = [
            r for r in dados_cotacao["value"]
            if ini <= datetime.strptime(r["dataHoraCotacao"][:10], "%Y-%m-%d").date() <= fim
            ]

if "value" not in dados_cotacao:
    raise ValueError("A resposta não contém a chave 'value'.")
if not dados_cotacao["value"]:
    raise ValueError(f"Nenhuma cotação para {data_inicio} a {data_fim}. Tente um intervalo maior.")
print(f"Registros retornados: {len(dados_cotacao['value'])}")


In [0]:
# REGRAS DE NEGÓCIO: A cotação da API é persistida em Delta com ingestion_datetime, seguindo o mesmo padrão de rastreabilidade das demais tabelas Bronze.
# Gravar o retorno da API na Bronze

df_cotacao = spark.createDataFrame(dados_cotacao["value"])
display(df_cotacao)
 
df_cotacao_bronze = (
    df_cotacao
    .withColumn("ingestion_datetime", current_timestamp())
)
 
display(df_cotacao_bronze)
 
(
    df_cotacao_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_cotacao_dolar")
)
  
display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_cotacao_dolar
        ORDER BY dataHoraCotacao
    """)
)
 
spark.sql("""
    DESCRIBE TABLE meu_catalog.bronze.tb_cotacao_dolar
""").show(truncate=False)
 
display(
    spark.sql("""
        SHOW TABLES IN meu_catalog.bronze
    """)
)


In [0]:
# Listar todas as tabelas da Bronze

spark.sql("""
    DESCRIBE TABLE meu_catalog.bronze.tb_cotacao_dolar
""").show(truncate=False)
 

display(
    spark.sql(f"SHOW TABLES IN {bronze_schema}")
)

In [0]:
# REGRAS DE NEGÓCIO: A validação final verifica se as tabelas Bronze possuem dados e timestamp de ingestão, reduzindo o risco de promover uma carga incompleta para a Silver.
# Conferir quantidade de registros e timestamp de ingestão

tabelas_bronze = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar"
]

for tabela in tabelas_bronze:
    resultado = spark.sql(f"""
        SELECT
            COUNT(*) AS quantidade_registros,
            MIN(ingestion_datetime) AS primeira_ingestao,
            MAX(ingestion_datetime) AS ultima_ingestao
        FROM {bronze_schema}.{tabela}
    """).collect()[0]

    print(
        f"{tabela}: "
        f"{resultado['quantidade_registros']} registros | "
        f"primeira ingestão: {resultado['primeira_ingestao']} | "
        f"última ingestão: {resultado['ultima_ingestao']}"
    )